## Example 1: geopandas functionality test 1

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon
import pandas as pd

# Example 1: Creating and working with GeoDataFrames
print("Example 1: Creating and working with GeoDataFrames")

try:
    # Create sample city data
    cities_data = {
        'city': ['San Francisco', 'Los Angeles', 'Seattle'],
        'state': ['CA', 'CA', 'WA'],
        'population': [873965, 3979576, 753675],
        'longitude': [-122.4194, -118.2437, -122.3321],
        'latitude': [37.7749, 34.0522, 47.6062]
    }
    
    # Create Point geometries
    geometry = [Point(xy) for xy in zip(cities_data['longitude'], cities_data['latitude'])]
    
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(cities_data, geometry=geometry, crs='EPSG:4326')
    
    print(f"✓ GeoDataFrame created:")
    print(f"  Shape: {gdf.shape}")
    print(f"  CRS: {gdf.crs}")
    print(f"  Geometry type: {gdf.geometry.geom_type.iloc[0]}")
    
    # Basic operations
    print(f"✓ Basic operations:")
    print(f"  Total population: {gdf['population'].sum():,}")
    print(f"  Bounds: {gdf.total_bounds}")
    print(f"  Area (degrees²): {gdf.geometry.area.sum():.6f}")
    
    print("✓ GeoDataFrame creation completed")
    
except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"geopandas example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 finished\n")

## Example 2: geopandas functionality test 2

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon

# Example 2: Spatial operations and relationships
print("Example 2: Spatial operations and relationships")

try:
    # Create points
    points_data = [
        ('Point A', Point(-122.4, 37.8)),
        ('Point B', Point(-122.3, 37.7)),
        ('Point C', Point(-122.5, 37.9))
    ]
    points_gdf = gpd.GeoDataFrame(points_data, columns=['name', 'geometry'], crs='EPSG:4326')
    
    # Create a polygon region
    polygon_coords = [(-122.6, 37.6), (-122.2, 37.6), (-122.2, 38.0), (-122.6, 38.0), (-122.6, 37.6)]
    region = gpd.GeoDataFrame([{'name': 'Region', 'geometry': Polygon(polygon_coords)}], crs='EPSG:4326')
    
    print(f"✓ Created {len(points_gdf)} points and {len(region)} polygon")
    
    # Spatial relationship tests
    within_region = points_gdf[points_gdf.geometry.within(region.geometry.iloc[0])]
    print(f"✓ Points within region: {len(within_region)}")
    
    # Buffer operations
    buffered_points = points_gdf.copy()
    buffered_points['geometry'] = points_gdf.geometry.buffer(0.1)  # 0.1 degree buffer
    print(f"✓ Created buffers around points")
    
    # Distance calculations
    distances = points_gdf.geometry.distance(region.geometry.iloc[0])
    print(f"✓ Distances to region boundary:")
    for name, dist in zip(points_gdf['name'], distances):
        print(f"  {name}: {dist:.4f} degrees")
    
    # Area calculations
    region_area = region.geometry.area.iloc[0]
    buffered_area = buffered_points.geometry.area.sum()
    print(f"✓ Region area: {region_area:.4f} degrees²")
    print(f"✓ Total buffered area: {buffered_area:.4f} degrees²")
    
    print("✓ Spatial operations completed")
    
except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"geopandas example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 2 finished\n")

## Example 3: geopandas functionality test 3

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, LineString
import pandas as pd

# Example 3: Coordinate reference system transformations
print("Example 3: Coordinate reference system transformations")

try:
    # Create GeoDataFrame with WGS84 coordinates (latitude/longitude)
    locations = {
        'place': ['Golden Gate Bridge', 'Alcatraz Island', 'Fishermans Wharf'],
        'type': ['bridge', 'island', 'attraction']
    }
    
    coords = [
        Point(-122.4783, 37.8199),  # Golden Gate Bridge
        Point(-122.4230, 37.8270),  # Alcatraz
        Point(-122.4106, 37.8080)   # Fishermans Wharf
    ]
    
    gdf_wgs84 = gpd.GeoDataFrame(locations, geometry=coords, crs='EPSG:4326')
    print(f"✓ Original CRS: {gdf_wgs84.crs}")
    
    # Project to UTM zone 10N (EPSG:32610, metres), the UTM zone that covers San Francisco
    gdf_utm = gdf_wgs84.to_crs('EPSG:32610')
    print(f"✓ Transformed CRS: {gdf_utm.crs}")
    assert gdf_utm.crs.to_epsg() == 32610
    
    # Compare coordinates
    print(f"✓ Coordinate comparison:")
    for i, place in enumerate(locations['place']):
        wgs84_coords = gdf_wgs84.geometry.iloc[i]
        utm_coords = gdf_utm.geometry.iloc[i]
        print(f"  {place}:")
        print(f"    WGS84: ({wgs84_coords.x:.4f}, {wgs84_coords.y:.4f})")
        print(f"    UTM 10N: ({utm_coords.x:.0f}, {utm_coords.y:.0f})")
    
    # Calculate distances in projected coordinates (more accurate)
    distances_deg = gdf_wgs84.geometry.distance(gdf_wgs84.geometry.iloc[0])
    distances_m = gdf_utm.geometry.distance(gdf_utm.geometry.iloc[0])
    
    print(f"✓ Distance calculations:")
    print(f"  In degrees: {distances_deg.iloc[1]:.6f}°")
    print(f"  In meters: {distances_m.iloc[1]:.0f} m")
    # Golden Gate Bridge to Alcatraz is a little under 5 km.
    assert 4500 < distances_m.iloc[1] < 5200, distances_m.iloc[1]
    
    print("✓ CRS transformation completed")
    
except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"geopandas example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 3 finished\n")

## Example 4: geopandas functionality test 4

In [ ]:
import geopandas as gpd
from shapely.geometry import Point, Polygon

# Example 4: Spatial joins and overlay operations
print("Example 4: Spatial joins and overlay operations")

try:
    # Create points of interest
    pois = gpd.GeoDataFrame([
        {'name': 'Restaurant A', 'type': 'restaurant', 'geometry': Point(-122.42, 37.77)},
        {'name': 'School B', 'type': 'school', 'geometry': Point(-122.41, 37.78)},
        {'name': 'Park C', 'type': 'park', 'geometry': Point(-122.43, 37.76)}
    ], crs='EPSG:4326')
    
    # Create neighborhood polygons
    neighborhoods = gpd.GeoDataFrame([
        {'name': 'Downtown', 'geometry': Polygon([(-122.44, 37.76), (-122.40, 37.76), 
                                                  (-122.40, 37.79), (-122.44, 37.79), (-122.44, 37.76)])},
        {'name': 'Mission', 'geometry': Polygon([(-122.44, 37.74), (-122.40, 37.74), 
                                                (-122.40, 37.76), (-122.44, 37.76), (-122.44, 37.74)])}
    ], crs='EPSG:4326')
    
    print(f"✓ Created {len(pois)} POIs and {len(neighborhoods)} neighborhoods")
    
    # Spatial join - find which neighborhood each POI is in
    joined = gpd.sjoin(pois, neighborhoods, how='left', predicate='within')
    print(f"✓ Spatial join results:")
    for _, row in joined.iterrows():
        neighborhood = row['name_right'] if pd.notna(row['name_right']) else 'Outside'
        print(f"  {row['name_left']} ({row['type']}) is in {neighborhood}")
    
    # Count POIs per neighborhood
    poi_counts = joined.groupby('name_right').size()
    print(f"✓ POI counts per neighborhood:")
    for neighborhood, count in poi_counts.items():
        print(f"  {neighborhood}: {count} POIs")
    
    # Create buffers around POIs
    buffered_pois = pois.copy()
    buffered_pois['geometry'] = pois.geometry.buffer(0.005)  # ~500m buffer
    
    # Find neighborhoods that intersect with POI buffers
    intersections = gpd.overlay(buffered_pois, neighborhoods, how='intersection')
    print(f"✓ Buffer intersections: {len(intersections)} areas")
    
    print("✓ Spatial joins and overlays completed")
    
except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"geopandas example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 4 finished\n")

## Example 5: geopandas functionality test 5

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# Example 5: Data aggregation and attribute operations
print("Example 5: Data aggregation and attribute operations")

try:
    # Create sample property data
    properties = gpd.GeoDataFrame([
        {'address': '123 Main St', 'price': 750000, 'bedrooms': 2, 'type': 'condo', 
         'geometry': Point(-122.42, 37.77)},
        {'address': '456 Oak Ave', 'price': 1200000, 'bedrooms': 3, 'type': 'house', 
         'geometry': Point(-122.41, 37.78)},
        {'address': '789 Pine Rd', 'price': 900000, 'bedrooms': 2, 'type': 'house', 
         'geometry': Point(-122.43, 37.76)},
        {'address': '321 Elm St', 'price': 650000, 'bedrooms': 1, 'type': 'condo', 
         'geometry': Point(-122.44, 37.79)}
    ], crs='EPSG:4326')
    
    print(f"✓ Created property dataset with {len(properties)} properties")
    
    # Basic statistics
    print(f"✓ Property statistics:")
    print(f"  Average price: ${properties['price'].mean():,.0f}")
    print(f"  Price range: ${properties['price'].min():,} - ${properties['price'].max():,}")
    print(f"  Total bedrooms: {properties['bedrooms'].sum()}")
    
    # Group by property type
    by_type = properties.groupby('type').agg({
        'price': ['mean', 'count'],
        'bedrooms': 'mean'
    }).round(0)
    
    print(f"✓ Analysis by property type:")
    for prop_type in properties['type'].unique():
        subset = properties[properties['type'] == prop_type]
        avg_price = subset['price'].mean()
        count = len(subset)
        print(f"  {prop_type.title()}: {count} properties, avg ${avg_price:,.0f}")
    
    # Spatial analysis
    centroid = properties.geometry.centroid
    print(f"✓ Spatial analysis:")
    print(f"  Centroid: ({centroid.x.mean():.4f}, {centroid.y.mean():.4f})")
    
    # Filter operations
    expensive = properties[properties['price'] > 800000]
    print(f"  Expensive properties (>$800k): {len(expensive)}")
    
    print("✓ Data aggregation and analysis completed")
    
except Exception as e:
    print(f"✗ Error: {e}")
    raise AssertionError(
        f"geopandas example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 5 finished\n")